### LSTM for predicting anamolies in EGG timeseries data (epileptic seizures)

Dataset from https://www.kaggle.com/datasets/harunshimanto/epileptic-seizure-recognition?resource=download

The original dataset from the reference consists of 5 different folders, each with 100 files, with each file representing a single subject/person. Each file is a recording of brain activity for 23.6 seconds. The corresponding time-series is sampled into 4097 data points. Each data point is the value of the EEG recording at a different point in time. So we have total 500 individuals with each has 4097 data points for 23.5 seconds.

We divided and shuffled every 4097 data points into 23 chunks, each chunk contains 178 data points for 1 second, and each data point is the value of the EEG recording at a different point in time. So now we have 23 x 500 = 11500 pieces of information(row), each information contains 178 data points for 1 second(column), the last column represents the label y {1,2,3,4,5}.

The response variable is y in column 179, the Explanatory variables X1, X2, …, X178

y contains the category of the 178-dimensional input vector. Specifically y in {1, 2, 3, 4, 5}:

5 - eyes open, means when they were recording the EEG signal of the brain the patient had their eyes open

4 - eyes closed, means when they were recording the EEG signal the patient had their eyes closed

3 - Yes they identify where the region of the tumor was in the brain and recording the EEG activity from the healthy brain area

2 - They recorded the EEG from the area where the tumor was located

1 - Recording of seizure activity

All subjects falling in classes 2, 3, 4, and 5 are subjects who did not have epileptic seizure. Only subjects in class 1 have epileptic seizure. Our motivation for creating this version of the data was to simplify access to the data via the creation of a .csv version of it. Although there are 5 classes most authors have done binary classification, namely class 1 (Epileptic seizure) against the rest.

This Dataset collect from UCI Machine Learning Repository

### Setup

In [14]:
import numpy as np
import torch

global_seed = 1066
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Set random seeds for reproducibility
np.random.seed(global_seed)
torch.manual_seed(global_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(global_seed)

print(f"Using device: {device}")

%matplotlib inline

Using device: mps


### Data Processing and Handling

In [15]:
import pandas as pd

# 1 row => 1 second of recording at 178 HZ 
# to generalize to longer formats we will use sliding window technique 

df = pd.read_csv('data.csv')
# internally used as an ID, we don't need it
df = df.drop("Unnamed", axis=1)

print(df.info()) 
print(f"DataFrame overview: {df.head()}") 


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11500 entries, 0 to 11499
Columns: 179 entries, X1 to y
dtypes: int64(179)
memory usage: 15.7 MB
None
DataFrame overview:     X1   X2   X3   X4   X5   X6   X7   X8   X9  X10  ...  X170  X171  X172  \
0  135  190  229  223  192  125   55   -9  -33  -38  ...   -17   -15   -31   
1  386  382  356  331  320  315  307  272  244  232  ...   164   150   146   
2  -32  -39  -47  -37  -32  -36  -57  -73  -85  -94  ...    57    64    48   
3 -105 -101  -96  -92  -89  -95 -102 -100  -87  -79  ...   -82   -81   -80   
4   -9  -65  -98 -102  -78  -48  -16    0  -21  -59  ...     4     2   -12   

   X173  X174  X175  X176  X177  X178  y  
0   -77  -103  -127  -116   -83   -51  4  
1   152   157   156   154   143   129  1  
2    19   -12   -30   -35   -35   -36  5  
3   -77   -85   -77   -72   -69   -65  5  
4   -32   -41   -65   -83   -89   -73  5  

[5 rows x 179 columns]


### Splitting The Data

In [16]:
from sklearn.model_selection import train_test_split

# our dataset is perfectly balanced, each class has 2300 entries
# but since we are binary classification our target is guessing y == 1, it'll be highlyunbalanced 
print(f"DataFrame shape: {df.groupby('y').size()}")

# 70 15 15 split
train_df, temp = train_test_split(
    df, 
    test_size=0.3, 
    random_state=global_seed, 
    stratify=df['y']
)

validation_df, test_df = train_test_split(
    temp, 
    test_size=0.5, 
    random_state=global_seed, 
    stratify=temp['y']
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training: {len(train_df)}")
print(f"Validation: {len(validation_df)}")
print(f"Test: {len(test_df)}")

DataFrame shape: y
1    2300
2    2300
3    2300
4    2300
5    2300
dtype: int64
Training: 8050
Validation: 1725
Test: 1725


### Data visualisation & Exploration

In [17]:
import plotly.graph_objects as go

class_titles = {
    1: "Active Seizure (1)",
    2: "Tumor Area (2)",
    3: "Healthy Area (3)",
    4: "Eyes Closed (4)",
    5: "Eyes Open (5)"
}

fig = go.Figure()

for item in class_titles:
    
    # one random sample from each class
    sample = train_df[train_df['y'] == item].sample(n=1)
    
    # convert to series
    voltages = sample.drop(columns=['y']).iloc[0]
    
    fig.add_trace(go.Scatter(
        x=voltages.index, 
        y=voltages.values, 
        mode='lines',
        name=class_titles.get(item, f"Class {item}") 
    ))

fig.update_layout(
    title="Overview of EEG Signals by Class",
    xaxis_title="Timestep (X1 - X178) - 1 second total",
    yaxis_title="Voltage",
    legend_title="EEG State",
    hovermode="x unified" 
)

fig.show()

In [18]:
import plotly.express as px

voltages = train_df.drop(columns=['y'])


plot_df = pd.DataFrame({
    "std_dev":  voltages.std(axis=1), # standard deviation for each row
    "class_name": train_df['y'].map(class_titles)
    })
print(f"Standard deviation of voltages (first 5 rows):\n{plot_df.head()}")

fig = px.box(plot_df,
             x="class_name", 
             y="std_dev", 
             labels={"class_name": "Patient State", 
                     "std_dev": "Standard Deviation"}, 
             title="Distribution of Standard Deviation by Patient State")
fig.show()

Standard deviation of voltages (first 5 rows):
     std_dev        class_name
0  47.142461     Eyes Open (5)
1  20.243939     Eyes Open (5)
2  25.083208  Healthy Area (3)
3  36.083253  Healthy Area (3)
4  74.310577   Eyes Closed (4)


we note that the standard deviation of the volage during and active seizure varies between 171 and 422 with a median of 278. This far dwarfs non-seizure voltage recorded. <br> We also note Tumor Area recordings have a significant amount of outlayers ranging from 130 to 541, overlapping with seizure activity.

Overall for **Binary Classification** the classes show good seperation. 

## LSTM Implemntation & Training

In [19]:
from sklearn.preprocessing import RobustScaler

# since we have lots of outliers we use the RobustScaler to normalize our data
scaler = RobustScaler()

data_cols = [col for col in train_df if col != "y"]

train_df[data_cols] = scaler.fit_transform(train_df[data_cols])

# we use transform instead of fit_transform to make sure the model normalize 
# the valdiation/test datasets with the same median and IQR as the training data
validation_df[data_cols] = scaler.transform(validation_df[data_cols])
test_df[data_cols] = scaler.transform(test_df[data_cols])



In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

 
class DatasetInit(Dataset):
    def __init__(self, dataframe):
        self.labels = dataframe['y'].values
        self.features = dataframe.drop(columns=['y']).values

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.features[idx], dtype=torch.float32) # int better?
        x_tensor = x_tensor.unsqueeze(-1) # LSTM expects (batch_size, seq_len, input_size)
        y_tensor = torch.tensor(1.0 if self.labels[idx] == 1 else 0.0, dtype=torch.long)
        
        return x_tensor, y_tensor
    

train_dataset = DatasetInit(dataframe=train_df)
vald_dataset = DatasetInit(dataframe=validation_df)
test_dataset = DatasetInit(dataframe=test_df)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
vald_loader = DataLoader(vald_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


### Model Architecture

In [21]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers    
        
        self.lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_size, # Good enough memory capacity?
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout
        )
        
        self.dropout = nn.Dropout(p=dropout)
        
        
        self.fc = nn.Linear(in_features=hidden_size, out_features=1)

    def forward(self, x):        
        # x => (64, 178, 1)
        
        # pass data through the LSTM
        lstm_out, (hidden_state, cell_state) = self.lstm(x)
        
        # we take the final timestep's output for classification
        final_out = lstm_out[:, -1, :]

        # fight overfitting with dropout, 0 for now
        x = self.dropout(final_out)
        
        # classify 
        x = self.fc(x)
        
        return x
    

model = LSTM(1, 64, 2).to(device)

# Hyperparameters
learning_rate = 0.001
epochs = 100
patience = 10 

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.25, patience=10)

# Binary Classification Loss
# do we need need label smoothing?
loss_fn = nn.BCEWithLogitsLoss()

## Training loop

In [ ]:
from tqdm import tqdm

train_loss = 0.0
best_f1_vald = 0.0
best_validatin_accuracy = 0.0
epochs_no_improve = 0

true_labels = []
raw_outputs = []

for epoch in range(epochs):
    # -- Training --
    model.train()
    
    train_tp = 0.0
    train_fp = 0.0
    train_fn = 0.0
    
    train_loop = tqdm(train_loader, desc="Training", leave=False)
    
    for batch_id, (x, y) in enumerate(train_loop):
        x = x.to(device)
        y = y.unsqueeze(1).float().to(device)
        # reset weights
        optimizer.zero_grad()
        # get ouput
        output = model(x)
        # accum loss
        loss = loss_fn(output, y)
        # backpropogate 
        loss.backward()
        # update weights
        optimizer.step()
        # accumulate loss
        train_loss += loss.item()
        
        prediction = (output >= 0.0).float() 
        # Our metric is going to be F1 score since our classes are not balanced
        train_tp += ((prediction == 1.0) & (y == 1.0)).sum().item()
        train_fp += ((prediction == 1.0) & (y == 0.0)).sum().item()
        train_fn += ((prediction == 0.0) & (y == 1.0)).sum().item()
    
    train_precision = train_tp / (train_tp + train_fp) if (train_tp + train_fp) > 0 else 0.0
    train_recall = train_tp / (train_tp + train_fn) if (train_tp + train_fn) > 0 else 0.0
    train_f1 = 2 * (train_precision * train_recall) / (train_precision + train_recall) if (train_precision + train_recall) > 0 else 0.0
    
    # -- Validation --
    model.eval()
    val_loss = 0.0
    vald_tp =vald_fp = vald_fn = vald_tn = 0
    validation_loop = tqdm(vald_loader, desc="Training", leave=False)

    with torch.no_grad():
        for x, y in validation_loop:
            x = x.to(device)
            y = y.unsqueeze(1).float().to(device)
            # prediction
            output = model(x)
            # calculate loss
            loss = loss_fn(output, y)
            # accumulate loss
            val_loss += loss.item()
            
            prediction = (output >= 0.0).float()          
            # F1 score
            vald_tp += ((prediction == 1.0) & (y == 1.0)).sum().item()
            vald_tn += ((prediction == 0.0) & (y == 0.0)).sum().item()
            vald_fp += ((prediction == 1.0) & (y == 0.0)).sum().item()
            vald_fn += ((prediction == 0.0) & (y == 1.0)).sum().item()
            # for graphing, .extend unpacks the list so we append correctly 
            raw_outputs.extend(output.cpu().numpy())
            true_labels.extend(y.cpu().numpy())
            
        avrg_vald_loss = val_loss / len(vald_loader)
        vald_precision = vald_tp / (vald_tp + vald_fp) if (vald_tp + vald_fp) > 0 else 0.0
        vald_recall = vald_tp / (vald_tp + vald_fn) if (vald_tp + vald_fn) > 0 else 0.0
        vald_f1 = 2 * (vald_precision * vald_recall) / (vald_precision + vald_recall) if (vald_precision + vald_recall) > 0 else 0.0
        
        # update learning rate
        scheduler.step(avrg_vald_loss)
        
    
        # -- Early stopping --
        if vald_f1 > best_f1_vald:
            best_f1_vald = vald_f1
            epochs_no_improve = 0
            # print(f"⭐ New best F1 Validation: {best_f1_vald}")
            # print(f"F1 Train: {train_f1}")
            # print(f"Epoch:{epoch} / {epochs - epoch}")
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Stopped after {epochs_no_improve} of no improvement")
            print(f"Best F1 Score {best_f1_vald:.2f}")
            break        


Stopped after 10 of no improvement
Best F1 Score 0.86


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(true_labels, raw_outputs)

df = pd.DataFrame({
    "False Positive Rate": fpr,
    "True Positive Rate": tpr
})

fig = px.line(
    df, 
    x="False Positive Rate", 
    y="True Positive Rate", 
    title=f"ROC Curve (Best F1: {best_f1_vald:.2f})", 
    range_x=[0, 1], 
    range_y=[0, 1]
)

# random guess line
fig.add_shape(
    type='line', line=dict(dash='dash', color='gray'),
    x0=0, x1=1, y0=0, y1=1
)

fig.show()
# the goal is to avoid false negatives (missing a seizure), 
# get all the true positives at the price of false positives